# GNN Stability Analysis: Baseline Results
## Cora Dataset - Edge Perturbation Experiments

**Author:** Senior Data Science Student  
**Supervisor:** Professor He  
**Date:** June 1, 2026  

This notebook analyzes the baseline GCN stability experiments on the Cora dataset under varying levels of edge perturbation.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Load Experimental Results

In [ ]:
# Load summary results
with open('../results/local_stability_metrics.json', 'r') as f:
    results = json.load(f)

# Extract perturbation levels and metrics
perturbation_levels = [0, 5, 10, 20]
mean_test_accs = []
std_test_accs = []
mean_val_accs = []
std_val_accs = []
mean_train_accs = []

for p in perturbation_levels:
    key = f'perturbation_{p}pct'
    mean_test_accs.append(results[key]['mean_test_accuracy'])
    std_test_accs.append(results[key]['std_test_accuracy'])
    mean_val_accs.append(results[key]['mean_val_accuracy'])
    std_val_accs.append(results[key]['std_val_accuracy'])
    mean_train_accs.append(results[key]['mean_train_accuracy'])

print("Loaded results for perturbation levels:", perturbation_levels)
print(f"Test accuracies: {[f'{acc*100:.2f}%' for acc in mean_test_accs]}")

## 2. Summary Statistics Table

In [ ]:
import pandas as pd

# Create summary dataframe
summary_df = pd.DataFrame({
    'Perturbation (%)': perturbation_levels,
    'Mean Test Acc (%)': [f'{acc*100:.2f}' for acc in mean_test_accs],
    'Std Test (%)': [f'{std*100:.2f}' for std in std_test_accs],
    'Mean Val Acc (%)': [f'{acc*100:.2f}' for acc in mean_val_accs],
    'Std Val (%)': [f'{std*100:.2f}' for std in std_val_accs],
    'Mean Train Acc (%)': [f'{acc*100:.2f}' for acc in mean_train_accs],
    'Delta from Baseline (%)': [f'{(acc - mean_test_accs[0])*100:.2f}' for acc in mean_test_accs]
})

print("\n=== Experimental Results Summary ===")
print(summary_df.to_string(index=False))

## 3. Perturbation Curve Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Convert to percentages
mean_test_pct = np.array(mean_test_accs) * 100
std_test_pct = np.array(std_test_accs) * 100
mean_val_pct = np.array(mean_val_accs) * 100
std_val_pct = np.array(std_val_accs) * 100

# Plot test accuracy with error bars
ax.errorbar(perturbation_levels, mean_test_pct, yerr=std_test_pct, 
            marker='o', markersize=8, linewidth=2, capsize=5, capthick=2,
            label='Test Accuracy', color='#2E86AB', alpha=0.9)

# Plot validation accuracy with error bars
ax.errorbar(perturbation_levels, mean_val_pct, yerr=std_val_pct, 
            marker='s', markersize=8, linewidth=2, capsize=5, capthick=2,
            label='Validation Accuracy', color='#A23B72', alpha=0.9)

# Formatting
ax.set_xlabel('Edge Perturbation Level (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_title('GCN Stability Under Edge Perturbations (Cora Dataset)', 
             fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=11, loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_xticks(perturbation_levels)

# Add baseline reference line
ax.axhline(y=mean_test_pct[0], color='gray', linestyle='--', alpha=0.5, linewidth=1.5, label='Baseline')

plt.tight_layout()
plt.savefig('../results/figures/perturbation_curve.png', dpi=300, bbox_inches='tight')
print("\n✅ Saved figure to: results/figures/perturbation_curve.png")
plt.show()

## 4. Variance Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Plot standard deviation trends
ax.plot(perturbation_levels, std_test_pct, marker='o', markersize=8, 
        linewidth=2.5, label='Test Std Dev', color='#2E86AB')
ax.plot(perturbation_levels, std_val_pct, marker='s', markersize=8, 
        linewidth=2.5, label='Val Std Dev', color='#A23B72')

ax.set_xlabel('Edge Perturbation Level (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Standard Deviation (%)', fontsize=13, fontweight='bold')
ax.set_title('Variance in GCN Performance Across Seeds', 
             fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(perturbation_levels)

plt.tight_layout()
plt.savefig('../results/figures/variance_analysis.png', dpi=300, bbox_inches='tight')
print("✅ Saved figure to: results/figures/variance_analysis.png")
plt.show()

## 5. Individual Seed Performance

In [ ]:
# Extract individual run data
seeds = [42, 101, 2023, 7, 88]
seed_data = {seed: [] for seed in seeds}

for p in perturbation_levels:
    key = f'perturbation_{p}pct'
    for run in results[key]['individual_runs']:
        seed_data[run['seed']].append(run['test_acc'] * 100)

# Plot individual seed trajectories
fig, ax = plt.subplots(figsize=(10, 6))

colors = plt.cm.viridis(np.linspace(0, 0.9, len(seeds)))
for idx, seed in enumerate(seeds):
    ax.plot(perturbation_levels, seed_data[seed], marker='o', 
            linewidth=1.5, alpha=0.7, label=f'Seed {seed}', color=colors[idx])

# Add mean line
ax.plot(perturbation_levels, mean_test_pct, marker='D', markersize=10,
        linewidth=3, color='black', label='Mean', linestyle='--')

ax.set_xlabel('Edge Perturbation Level (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_title('Individual Seed Performance Trajectories', 
             fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
ax.set_xticks(perturbation_levels)

plt.tight_layout()
plt.savefig('../results/figures/seed_trajectories.png', dpi=300, bbox_inches='tight')
print("✅ Saved figure to: results/figures/seed_trajectories.png")
plt.show()

## 6. Train/Val/Test Gap Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

mean_train_pct = np.array(mean_train_accs) * 100

# Plot all three splits
ax.plot(perturbation_levels, mean_train_pct, marker='^', markersize=8,
        linewidth=2.5, label='Train Accuracy', color='#06A77D')
ax.plot(perturbation_levels, mean_val_pct, marker='s', markersize=8,
        linewidth=2.5, label='Validation Accuracy', color='#A23B72')
ax.plot(perturbation_levels, mean_test_pct, marker='o', markersize=8,
        linewidth=2.5, label='Test Accuracy', color='#2E86AB')

ax.set_xlabel('Edge Perturbation Level (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_title('Train/Val/Test Performance Under Perturbations', 
             fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(perturbation_levels)

plt.tight_layout()
plt.savefig('../results/figures/train_val_test_gap.png', dpi=300, bbox_inches='tight')
print("✅ Saved figure to: results/figures/train_val_test_gap.png")
plt.show()

## 7. Key Observations

### Performance Degradation
- **Baseline (0%):** 79.70% ± 0.45%
- **5% perturbation:** 79.46% ± 0.63% (Δ = -0.24%)
- **10% perturbation:** 78.72% ± 0.92% (Δ = -0.98%)
- **20% perturbation:** 77.94% ± 0.59% (Δ = -1.76%)

### Variance Analysis
- Standard deviation **increases** at 10% perturbation (0.92%), suggesting an **instability threshold**
- Variance drops slightly at 20%, possibly due to structural collapse into a more uniform degraded state

### Train/Val/Test Gaps
- Train accuracy remains high (~99%) across all perturbation levels
- Validation and test accuracies track closely, suggesting good generalization
- Gap between train and test widens slightly with perturbation, indicating potential overfitting to corrupted structure

### Seed Consistency
- All seeds show similar degradation patterns
- Some seeds (e.g., Seed 42) consistently perform slightly below mean
- No outlier seeds detected

## 8. Next Steps

1. **Finer granularity:** Add 2.5%, 7.5%, 15% perturbation levels to better characterize the 5-10% transition zone
2. **Spectral analysis:** Compute eigenvalue distributions of perturbed Laplacians
3. **Lipschitz bounds:** Measure empirical Lipschitz constants
4. **Architecture comparison:** Test GAT and other architectures
5. **Regularization:** Implement Lipschitz-constrained variants

## 9. Export Summary for README

In [ ]:
# Generate markdown table for README
print("\n=== Markdown Table for README ===")
print("| Perturbation Level | Mean Test Acc | Std Dev | Mean Val Acc | Delta from Baseline |")
print("|-------------------|---------------|---------|--------------|---------------------|")
for i, p in enumerate(perturbation_levels):
    print(f"| {p}% | {mean_test_pct[i]:.2f}% | ±{std_test_pct[i]:.2f}% | {mean_val_pct[i]:.2f}% | {(mean_test_pct[i] - mean_test_pct[0]):.2f}% |")